# 02 — Inventory & Calculation

**Rules:** PEP ecopassport PCR ed4 (S08) and PSR-0018 ed1.3 (S01), as set in `01_scope_boundary`.

This notebook turns the decisions in notebook 01 into numbers: the life cycle emissions of one wallbox (declared unit, for CDR sizing), the emissions per kWh supplied (functional unit, for PEP benchmarking), and the share of each stage (hotspots).

**How to use:** Fill in every cell marked `TODO`, in order. Every number must have a source ID (`data/raw/SOURCES.md`) or an assumption ID (`docs/assumptions_log.md`). Calculation cells produce results only after the inputs above them are filled in.

## 0. Parameters from notebook 01

These values were decided in notebook 01. Do not change them here; change notebook 01 first if a decision changes.

In [29]:
SERVICE_LIFE_YEARS = 10                 # S01, p. 20
ENERGY_PER_CHARGING_POINT_KWH = 28_251  # S01, p. 20 (derivation in notebook 01, Section 2)
CHARGING_POINTS = 1                     # Notebook 01, reference flow
CABLE_LENGTH_FT = 24                    # A02 (sensitivity range 13.1-25 ft)
FT_TO_M = 0.3048                        # Exact unit conversion
CUTOFF_THRESHOLD = 0.05                 # S08, §2.3 (aggregate mass, energy, impact)
DQR_RELEVANCE_SHARE = 0.80              # S08, §2.9.3 (score datasets covering >= 80% of impact)
DEFAULT_DQR = 5                         # S08, §2.9.3 (when quality cannot be assessed)

ENERGY_SUPPLIED_KWH = ENERGY_PER_CHARGING_POINT_KWH * CHARGING_POINTS
REFERENCE_FLOW = 1 / ENERGY_SUPPLIED_KWH  # wallboxes per kWh (functional unit)
CABLE_LENGTH_M = CABLE_LENGTH_FT * FT_TO_M

print(f"Energy supplied over service life: {ENERGY_SUPPLIED_KWH:,} kWh")
print(f"Reference flow: {REFERENCE_FLOW:.4e} wallboxes per kWh")
print(f"Cable length: {CABLE_LENGTH_M:.2f} m")

Energy supplied over service life: 28,251 kWh
Reference flow: 3.5397e-05 wallboxes per kWh
Cable length: 7.32 m


## 1. Data Sources

**Guiding questions**
- Which public databases or datasets can provide emission factors for the materials in a wallbox? (ecoinvent requires a paid license, so it cannot be used.)
- Where does each candidate rank in the PCR ed4 source priority order (S08, §2.9)?
- Which candidate offers global average (GLO) datasets, as required by A03?

In [30]:
import pandas as pd

# PCR ed4 source priority (S08, §2.9): 1 = verified with DQR ... 4 = ISO 14040/44 study with independent
# verification ... 6 = no proof of verification (selection must be justified).
data_sources = pd.DataFrame(
    [
        {
            "source_id": "S09",
            "name": "Copper and Copper Alloy Semi-Fabricated Products LCA",
            "publisher": "International Copper Association (ICA)",
            "public_access": True,
            "pcr_priority_level": 4,
            "glo_datasets": "Global (excludes China)",
            "used_for": "Copper",
            "notes": "Copper wire rod; ends before wire drawing, which must be added separately",
        },
        {
            "source_id": "S18",
            "name": "Eco-profile of Polyamide 6 (PA6)",
            "publisher": "PlasticsEurope",
            "public_access": True,
            "pcr_priority_level": 4,
            "glo_datasets": "No (Europe)",
            "used_for": "PA",
            "notes": "Critically reviewed; EF 3.0; represents ~21% of EU PA6 capacity",
        },
        {
            "source_id": "S19",
            "name": "UK Government GHG Conversion Factors 2026 (Material use)",
            "publisher": "UK DESNZ",
            "public_access": True,
            "pcr_priority_level": 6,
            "glo_datasets": "No (UK)",
            "used_for": "PC proxy (A10), PVC, steel, aluminium, cardboard, paper, wood, PCB proxy",
            "notes": "Government factor set, not a verified LCI; chosen for consistency and public access. "
                     "PC and PCB are category proxies (average rigid plastic; IT electrical items)",
        },
        {
            "source_id": "S20",
            "name": "eGRID2023 (2023 data), rev2",
            "publisher": "U.S. EPA",
            "public_access": True,
            "pcr_priority_level": 6,
            "glo_datasets": "No (U.S. states)",
            "used_for": "Use-stage electricity, California vs. Texas",
            "notes": "Official U.S. grid emission rates; not an LCI module, so no PCR verification level applies",
        },
    ]
)
data_sources

,source_id,name,publisher,public_access,pcr_priority_level,glo_datasets,used_for,notes
0,S09,Copper and Copper Alloy Semi-Fabricated Produc...,International Copper Association (ICA),True,4,Global (excludes China),Copper,"Copper wire rod; ends before wire drawing, whi..."
1,S18,Eco-profile of Polyamide 6 (PA6),PlasticsEurope,True,4,No (Europe),PA,Critically reviewed; EF 3.0; represents ~21% o...
2,S19,UK Government GHG Conversion Factors 2026 (Mat...,UK DESNZ,True,6,No (UK),"PC proxy (A10), PVC, steel, aluminium, cardboa...","Government factor set, not a verified LCI; cho..."
3,S20,"eGRID2023 (2023 data), rev2",U.S. EPA,True,6,No (U.S. states),"Use-stage electricity, California vs. Texas",Official U.S. grid emission rates; not an LCI ...


## 2. Reference Product and Bill of Materials (BOM)

With public data only, the BOM is a **proxy BOM** built with a data hierarchy:

| Tier | Data | Use in this project |
|---|---|---|
| 1 | Manufacturer's actual BOM | Not available (public data only) |
| 2 | Material composition from published PEPs or EPDs of similar wallboxes | Main source of material shares |
| 3 | Engineering estimate | Cable copper from length × gauge |
| 4 | Published LCA studies of EV chargers | Fill remaining gaps |
| 5 | Spend-based estimate | Screening only, last resort |

**Guiding questions**
- Which reference product has a public total mass (S04–S06)? Does that mass include the cable and the mounting bracket?
- Which published PEP or EPD of a wallbox can provide material shares?
- Which PCR ed4 Table 2 items (copper, gold, silver, microprocessors) appear, and where?
- Does the sum of all components equal the total product mass (mass balance)?

### 2a. How the proxy BOM is built

No manufacturer BOM is public, so the BOM is assembled from four parts, each with its own source:

| Part | Mass | How it is derived |
|---|---|---|
| Body (everything except cable, connector, packaging) | 3.6 kg (A04) | Median of sources that state a mass without the cable; split into materials with the mean shares of five PEPs (S10–S14) |
| Charging cable | ≈ 3.0 kg (A05) | Copper from conductor size × length × copper density; jacket scaled from S13 |
| Connector (charging handle) | 0.54 kg (A06) | Manufacturer weight of a J1772 connector (S17); modeled with the copper factor (A11) |
| Packaging | 2.05 kg (A09) | Median of the five PEPs; split with their mean shares |

### 2b. Body: material shares from five PEPs (A04, A07, A08)

Each PEP lists materials for the whole product. To use them for the body only, the charging cable and packaging are removed and each PEP is converted to shares of its own body. The mean of the five PEPs is then applied to the 3.6 kg body.

Grouping rules:
- PC and ABS are combined as housing PC (A07).
- PCB = "electronic cards" (S10, S11), "PWB" (S12), "electronic components" (S14), epoxy + glass + TBBPA (S13). The S12 and S13 values may understate the PCB (A08).
- S13 does not report its cable separately, so all its copper and PVC are assigned to the cable (A05 note); its body copper is only the brass.

In [31]:
BODY_MASS_KG = 3.6  # A04

# Published body materials (cable and packaging excluded).
# S10, S11, S12, S14: % of total mass incl. packaging (p. 2); S13: grams (p. 2). Only the ratios are used.
pep_body = pd.DataFrame({
    "S10": {"PC": 25.0, "ABS": 1.4, "PA": 2.5, "PBT": 0.4, "Other plastics": 0.4 + 0.5, "Steel": 3.2,
            "Copper": 1.0, "Aluminium": 0.1, "PCB": 8.4, "Internal wiring": 1.7, "Other components": 0},
    "S11": {"PC": 21.7, "ABS": 1.2, "PA": 3.3, "PBT": 0.7, "Other plastics": 0.6 + 0.2, "Steel": 3.8 + 0.4,
            "Copper": 1.4, "Aluminium": 0, "PCB": 8.8, "Internal wiring": 0.8, "Other components": 0},
    "S12": {"PC": 18.0, "ABS": 0, "PA": 5.8, "PBT": 0.8, "Other plastics": 2.6 + 0.8 + 0.7, "Steel": 13.0 + 0.4,
            "Copper": 3.4, "Aluminium": 0.1, "PCB": 1.7 + 0.4, "Internal wiring": 0, "Other components": 1.9 + 0.1},
    "S13": {"PC": 3012.00, "ABS": 0, "PA": 0, "PBT": 0, "Other plastics": 36.71 + 36.03 + 102.68, "Steel": 68.51 + 52.08,
            "Copper": 40.38, "Aluminium": 35.17 + 115.54, "PCB": 133.13 + 207.79 + 20.98, "Internal wiring": 0, "Other components": 100.68},
    "S14": {"PC": 1.1, "ABS": 21.7, "PA": 5.6, "PBT": 0, "Other plastics": 0.2, "Steel": 1.6,
            "Copper": 1.7 + 0.1, "Aluminium": 0.4, "PCB": 10.7, "Internal wiring": 0, "Other components": 0},
}).T

body_shares = pep_body.div(pep_body.sum(axis=1), axis=0)   # each PEP as shares of its own body
mean_shares = body_shares.mean()                            # mean of the five PEPs
mean_shares["PC"] += mean_shares.pop("ABS")                 # A07: housing modeled as PC
body_materials = (mean_shares * BODY_MASS_KG).sort_values(ascending=False)

pd.DataFrame({"share": mean_shares.reindex(body_materials.index).round(3), "mass_kg": body_materials.round(3)})

,share,mass_kg
PC,0.555,2.000
PCB,0.155,0.558
Steel,0.101,0.365
PA,0.076,0.273
Copper,0.035,0.126
Other plastics,0.034,0.123
Other components,0.013,0.047
Internal wiring,0.011,0.041
Aluminium,0.010,0.037
PBT,0.008,0.030


### 2c. Charging cable (A05)

Copper mass = conductor cross-section × cable length × copper density, for each conductor:
- 2 power conductors at 8 AWG (S07), 1 ground at 10 AWG and 1 signal at 18 AWG (A05)
- Cross-sections are the standard AWG values; copper density is 8.96 g/cm³

Jacket and insulation = 204 g per metre (S13: 1,326.30 g of PVC on a 6.5 m cable) × cable length.

In [32]:
COPPER_DENSITY_KG_M3 = 8960                           # physical constant
AWG_AREA_MM2 = {8: 8.367, 10: 5.261, 18: 0.823}       # standard AWG cross-sections
CABLE_CONDUCTORS_AWG = [8, 8, 10, 18]                 # A05: 2 power (S07), 1 ground, 1 signal
JACKET_KG_PER_M = 1.32630 / 6.5                       # S13

cable_copper_kg = sum(AWG_AREA_MM2[g] * 1e-6 * CABLE_LENGTH_M * COPPER_DENSITY_KG_M3 for g in CABLE_CONDUCTORS_AWG)
cable_jacket_kg = JACKET_KG_PER_M * CABLE_LENGTH_M
print(f"Cable copper: {cable_copper_kg:.3f} kg | Jacket: {cable_jacket_kg:.3f} kg | Cable: {cable_copper_kg + cable_jacket_kg:.3f} kg")

Cable copper: 1.496 kg | Jacket: 1.493 kg | Cable: 2.988 kg


### 2d. Connector and packaging (A06, A09)

- Connector: 540 g (S17). Its plastic and copper split is not public; it is modeled with the copper factor, the more conservative choice (A11).
- Packaging: median mass of the five PEPs, split with their mean shares of cardboard, wood, and paper.

In [33]:
CONNECTOR_MASS_KG = 0.54  # A06, S17

# Packaging as published (same units as pep_body); S12 and S14 list no wood or no paper.
pep_packaging = pd.DataFrame({
    "S10": {"Cardboard": 19.6, "Wood": 6.6, "Paper": 0.8},
    "S11": {"Cardboard": 16.8, "Wood": 5.7, "Paper": 0.7},
    "S12": {"Cardboard": 10.4, "Wood": 10.7, "Paper": 1.8},
    "S13": {"Cardboard": 1352.86, "Wood": 183.57, "Paper": 0},
    "S14": {"Cardboard": 16.2, "Wood": 0, "Paper": 0.0},
}).T
PACKAGING_MASS_KG = pd.Series({"S10": 2.06, "S11": 2.05, "S12": 2.17, "S13": 1.54, "S14": 1.28}).median()  # A09
packaging_shares = pep_packaging.div(pep_packaging.sum(axis=1), axis=0).mean()
packaging_materials = packaging_shares * PACKAGING_MASS_KG
pd.DataFrame({"share": packaging_shares.round(3), "mass_kg": packaging_materials.round(3)})

,share,mass_kg
Cardboard,0.757,1.552
Wood,0.215,0.442
Paper,0.028,0.057


### 2e. BOM table

`data_tier` follows the table in Section 2: 2 = shares from published PEPs, 3 = engineering estimate or proxy product. `table2_item` marks PCR ed4 Table 2 materials (copper; the PCB contains gold, silver, and microprocessors).

In [34]:
rows = []
body_note = "3.6 kg body (A04) x mean share of five PEPs"
for material, mass in body_materials.items():
    rows.append({"component": "Body", "material": material, "mass_kg": mass,
                 "source_id": "S10-S14; A04", "data_tier": 2,
                 "table2_item": material in ("PCB", "Copper", "Internal wiring"),
                 "derivation": body_note + (" (PC + ABS, A07)" if material == "PC" else " (A08)" if material == "PCB" else "")})
rows += [
    {"component": "Charging cable", "material": "Copper", "mass_kg": cable_copper_kg, "source_id": "S07; A02; A05",
     "data_tier": 3, "table2_item": True, "derivation": "Cross-section x 7.32 m x 8.96 g/cm3 for 2x8, 1x10, 1x18 AWG"},
    {"component": "Charging cable", "material": "PVC", "mass_kg": cable_jacket_kg, "source_id": "S13; A02; A05",
     "data_tier": 3, "table2_item": False, "derivation": "204 g/m (S13 PVC / 6.5 m) x 7.32 m"},
    {"component": "Connector", "material": "Connector", "mass_kg": CONNECTOR_MASS_KG, "source_id": "S17; A06",
     "data_tier": 3, "table2_item": True, "derivation": "Manufacturer weight of a J1772 demo connector; modeled as copper (A11)"},
]
for material, mass in packaging_materials.items():
    rows.append({"component": "Packaging", "material": material, "mass_kg": mass, "source_id": "S10-S14; A09",
                 "data_tier": 2, "table2_item": False, "derivation": "2.05 kg packaging (A09) x mean share of five PEPs"})

bom = pd.DataFrame(rows)
bom["mass_kg"] = bom["mass_kg"].round(3)
bom

,component,material,mass_kg,source_id,data_tier,table2_item,derivation
0,Body,PC,2.000,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs (P...
1,Body,PCB,0.558,S10-S14; A04,2,True,3.6 kg body (A04) x mean share of five PEPs (A08)
2,Body,Steel,0.365,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs
3,Body,PA,0.273,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs
4,Body,Copper,0.126,S10-S14; A04,2,True,3.6 kg body (A04) x mean share of five PEPs
5,Body,Other plastics,0.123,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs
6,Body,Other components,0.047,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs
7,Body,Internal wiring,0.041,S10-S14; A04,2,True,3.6 kg body (A04) x mean share of five PEPs
8,Body,Aluminium,0.037,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs
9,Body,PBT,0.030,S10-S14; A04,2,False,3.6 kg body (A04) x mean share of five PEPs


### 2f. Mass balance and plausibility

1. **Mass balance:** the BOM rows must add up to the component masses they were derived from.
2. **Plausibility:** the product mass is compared with independent figures. ChargePoint (S04) and Tesla (S05) totals are only a cross-check, because their datasheets do not state whether the total includes the cable.

In [35]:
product = bom[bom["component"] != "Packaging"]["mass_kg"].sum()
expected = BODY_MASS_KG + cable_copper_kg + cable_jacket_kg + CONNECTOR_MASS_KG
REFERENCE_PRODUCT_MASS_KG = round(expected, 3)  # product without packaging, used for the cut-off check in Section 9

print(f"BOM product total: {product:.3f} kg | Sum of component masses: {expected:.3f} kg | Gap: {product - expected:+.3f} kg")
print(f"BOM total incl. packaging: {bom['mass_kg'].sum():.2f} kg (five PEPs: 7.62-9.38 kg incl. packaging)")
print(f"Body + cable without connector: {BODY_MASS_KG + cable_copper_kg + cable_jacket_kg:.2f} kg "
      f"(ChargePoint 6.3 kg, S04; Tesla 6.8 kg incl. bracket, S05)")
bom.groupby("component")["mass_kg"].sum().round(3)

BOM product total: 7.129 kg | Sum of component masses: 7.128 kg | Gap: +0.001 kg
BOM total incl. packaging: 9.18 kg (five PEPs: 7.62-9.38 kg incl. packaging)
Body + cable without connector: 6.59 kg (ChargePoint 6.3 kg, S04; Tesla 6.8 kg incl. bracket, S05)


component
Body              3.600
Charging cable    2.989
Connector         0.540
Packaging         2.051
Name: mass_kg, dtype: float64

## 3. Emission Factors and Data Quality (DQR)

**Guiding questions**
- For each material in the BOM, which dataset gives its emission factor (kg CO₂e per kg)?
- How well does each dataset match in technology (TeR), geography (GR), and time (TiR), on a scale from 1 (best) to 5 (worst)?
- Which datasets together cover at least 80% of the climate change impact, so they must be scored?

**Factor choices** (sources selected in Section 1):

| BOM material | Factor | Source | Note |
|---|---|---|---|
| Copper, internal wiring, connector | Copper wire rod, global | S09 | Wiring and connector modeled as copper (A11); wire drawing handled in Section 9 (A13) |
| PA | PA6 eco-profile | S18 | Europe |
| PC, PBT, other plastics | Average rigid plastic | S19 | PC proxy (A10) |
| PVC | PVC incl. forming | S19 | |
| PCB | Electrical items – IT | S19 | Category proxy |
| Other components | Electrical items – small | S19 | A12 |
| Steel / Aluminium | Steel cans / Aluminium cans and foil | S19 | Category proxies |
| Cardboard, paper, wood | Board, paper, wood | S19 | |

**DQR scoring** follows PCR ed4 Annex E (S08, p. 68): TiR 1 = study within the dataset's validity, +1 per 2 years after it; TeR 1 = same technology … 4 = similar … 5 = different; GR 1 = same country … 4 = different region judged similar … 5 = different. The study location is a global average (A03), so UK or European datasets score GR 4.

In [36]:
# Factors in kg CO2e per kg of material (cradle-to-gate). DESNZ values are per tonne / 1000.
ef_rows = [
    # material,          factor,           dataset,                                   source, TiR, TeR, GR, DQR note
    ("Copper",           3622 / 1000,     "ICA copper wire rod, global (EF 3.0)",     "S09", 2, 3, 3, "Data 2019-21; wire rod, not drawn wire; global excl. China"),
    ("Internal wiring",  3622 / 1000,     "ICA copper wire rod (A11)",                "S09", 2, 3, 3, "Modeled as copper"),
    ("Connector",        3622 / 1000,     "ICA copper wire rod (A11)",                "S09", 2, 4, 3, "Conservative proxy for a plastic and copper part"),
    ("PA",               4.52,            "PlasticsEurope PA6 eco-profile (EF 3.0)",  "S18", 1, 2, 4, "Valid to 2027; Europe; ~21% of EU capacity"),
    ("PC",               3351.96455 / 1000, "DESNZ average plastic rigid (A10)",      "S19", 1, 4, 4, "PC proxy; UK factor set"),
    ("PBT",              3351.96455 / 1000, "DESNZ average plastic rigid",            "S19", 1, 4, 4, "Proxy; UK factor set"),
    ("Other plastics",   3351.96455 / 1000, "DESNZ average plastic rigid",            "S19", 1, 4, 4, "Proxy; UK factor set"),
    ("PVC",              2942.43735 / 1000, "DESNZ PVC (incl. forming)",              "S19", 1, 2, 4, "UK factor set"),
    ("PCB",              24865.47556 / 1000, "DESNZ electrical items - IT",           "S19", 1, 4, 4, "Whole IT products, not PCB only"),
    ("Other components", 5647.94563 / 1000, "DESNZ electrical items - small (A12)",   "S19", 1, 4, 4, "Proxy for relays and similar parts"),
    ("Steel",            2861.58251 / 1000, "DESNZ metal: steel cans",                "S19", 1, 4, 4, "Can steel as proxy for steel parts"),
    ("Aluminium",        9113.58251 / 1000, "DESNZ metal: aluminium cans and foil",   "S19", 1, 4, 4, "Proxy for aluminium parts"),
    ("Cardboard",        1198.23866 / 1000, "DESNZ paper and board: board",           "S19", 1, 2, 4, "UK factor set"),
    ("Paper",            1343.59114 / 1000, "DESNZ paper and board: paper",           "S19", 1, 2, 4, "UK factor set"),
    ("Wood",             269.50416 / 1000,  "DESNZ wood (construction)",              "S19", 1, 3, 4, "Construction wood as proxy for pallet wood"),
]
emission_factors = pd.DataFrame(ef_rows, columns=["material", "ef_kgco2e_per_kg", "dataset", "source_id", "TiR", "TeR", "GR", "dqr_note"])
emission_factors["DQR"] = emission_factors[["TeR", "GR", "TiR"]].mean(axis=1).round(2)

# Every BOM material needs a factor
missing = set(bom["material"]) - set(emission_factors["material"])
print("BOM materials without a factor:", missing or "none")

# Which datasets cover >= 80% of the material impact (these must be scored, S08 §2.9.3)
impact = bom.merge(emission_factors[["material", "ef_kgco2e_per_kg", "DQR"]], on="material")
impact["kgco2e"] = impact["mass_kg"] * impact["ef_kgco2e_per_kg"]
by_material = impact.groupby("material").agg(kgco2e=("kgco2e", "sum"), DQR=("DQR", "first")).sort_values("kgco2e", ascending=False)
by_material["share"] = by_material["kgco2e"] / by_material["kgco2e"].sum()
by_material["cumulative"] = by_material["share"].cumsum()
by_material["relevant_80pct"] = by_material["cumulative"].shift(fill_value=0) < DQR_RELEVANCE_SHARE
print(by_material.round(3).to_string())
emission_factors

BOM materials without a factor: none
                  kgco2e   DQR  share  cumulative  relevant_80pct
material                                                         
PCB               13.875  3.00  0.361       0.361            True
PC                 6.704  3.00  0.175       0.536            True
Copper             5.875  2.67  0.153       0.689            True
PVC                4.393  2.33  0.114       0.803            True
Connector          1.956  3.00  0.051       0.854           False
Cardboard          1.860  2.33  0.048       0.903           False
PA                 1.234  2.33  0.032       0.935           False
Steel              1.044  3.00  0.027       0.962           False
Other plastics     0.412  3.00  0.011       0.973           False
Aluminium          0.337  3.00  0.009       0.982           False
Other components   0.265  3.00  0.007       0.988           False
Internal wiring    0.149  2.67  0.004       0.992           False
Wood               0.119  2.67  0.003  

,material,ef_kgco2e_per_kg,dataset,source_id,TiR,TeR,GR,dqr_note,DQR
0,Copper,3.622000,"ICA copper wire rod, global (EF 3.0)",S09,2,3,3,"Data 2019-21; wire rod, not drawn wire; global...",2.67
1,Internal wiring,3.622000,ICA copper wire rod (A11),S09,2,3,3,Modeled as copper,2.67
2,Connector,3.622000,ICA copper wire rod (A11),S09,2,4,3,Conservative proxy for a plastic and copper part,3.00
3,PA,4.520000,PlasticsEurope PA6 eco-profile (EF 3.0),S18,1,2,4,Valid to 2027; Europe; ~21% of EU capacity,2.33
4,PC,3.351965,DESNZ average plastic rigid (A10),S19,1,4,4,PC proxy; UK factor set,3.00
5,PBT,3.351965,DESNZ average plastic rigid,S19,1,4,4,Proxy; UK factor set,3.00
6,Other plastics,3.351965,DESNZ average plastic rigid,S19,1,4,4,Proxy; UK factor set,3.00
7,PVC,2.942437,DESNZ PVC (incl. forming),S19,1,2,4,UK factor set,2.33
8,PCB,24.865476,DESNZ electrical items - IT,S19,1,4,4,"Whole IT products, not PCB only",3.00
9,Other components,5.647946,DESNZ electrical items - small (A12),S19,1,4,4,Proxy for relays and similar parts,3.00


## 4. Manufacturing Stage (A1–A3)

Manufacturing = materials (mass × factor) + manufacturing waste.

**What the factors already include:** DESNZ material factors (S19) cover raw material extraction through to a finished good, including forming (for example moulding of plastics) and transport between stages (S19 methodology report, §12.15 and §12.19, pp. 122–123). They use IPCC AR5 GWPs (§1.9, p. 14), the same basis as EF 3.0 used by S09 and S18.

**Gaps handled elsewhere:** wire drawing of the cable copper (A13), moulding of PA (S18 ends at pellets), and final assembly energy (A15) are checked against the 5% cut-off in Section 9.

**Manufacturing waste (A14):** PSR-0018 §3.5.1.1 (S01, p. 17) sets a default when waste treatment cannot be substantiated: waste equal to 30% of the product mass, landfilled after 1,000 km by lorry. The extra material that becomes waste is also counted (conservative reading).

In [37]:
MFG_WASTE_RATE = 0.30                  # S01 p. 17 (A14)
WASTE_TRANSPORT_KM = 1000              # S01 p. 17
HGV_KGCO2E_PER_TKM = 0.10356           # S19: HGV, average non-refrigerated, average laden
LANDFILL_KGCO2E_PER_T = 9.00687        # S19: landfill of plastics, metals, and WEEE

manufacturing = bom.merge(emission_factors[["material", "ef_kgco2e_per_kg"]], on="material", how="left")
missing = manufacturing[manufacturing["ef_kgco2e_per_kg"].isna()]["material"].unique()
if len(missing):
    print("Materials without an emission factor:", list(missing))
manufacturing["kgco2e"] = manufacturing["mass_kg"] * manufacturing["ef_kgco2e_per_kg"]

# Manufacturing waste (A14): based on the product only, not the packaging
product_rows = manufacturing["component"] != "Packaging"
waste_kg = MFG_WASTE_RATE * manufacturing.loc[product_rows, "mass_kg"].sum()
waste_rows = pd.DataFrame([
    {"component": "Manufacturing waste", "material": "Extra material (30% of product materials)",
     "mass_kg": waste_kg, "kgco2e": MFG_WASTE_RATE * manufacturing.loc[product_rows, "kgco2e"].sum()},
    {"component": "Manufacturing waste", "material": "Transport to landfill (1,000 km lorry)",
     "mass_kg": waste_kg, "kgco2e": waste_kg / 1000 * WASTE_TRANSPORT_KM * HGV_KGCO2E_PER_TKM},
    {"component": "Manufacturing waste", "material": "Landfill",
     "mass_kg": waste_kg, "kgco2e": waste_kg / 1000 * LANDFILL_KGCO2E_PER_T},
])
manufacturing = pd.concat([manufacturing, waste_rows], ignore_index=True)

summary = manufacturing.groupby("component")["kgco2e"].sum()
print(summary.round(2).to_string())
print(f"Manufacturing total: {manufacturing['kgco2e'].sum():.2f} kg CO2e per wallbox "
      f"(PEP benchmark S10-S13: 41-126 kg CO2e)")
manufacturing[["component", "material", "mass_kg", "kgco2e"]].round(3)

component
Body                   24.58
Charging cable          9.81
Connector               1.96
Manufacturing waste    11.14
Packaging               2.06
Manufacturing total: 49.54 kg CO2e per wallbox (PEP benchmark S10-S13: 41-126 kg CO2e)


,component,material,mass_kg,kgco2e
0,Body,PC,2.000,6.704
1,Body,PCB,0.558,13.875
2,Body,Steel,0.365,1.044
3,Body,PA,0.273,1.234
4,Body,Copper,0.126,0.456
5,Body,Other plastics,0.123,0.412
6,Body,Other components,0.047,0.265
7,Body,Internal wiring,0.041,0.149
8,Body,Aluminium,0.037,0.337
9,Body,PBT,0.030,0.101


## 5. Distribution (A4) and Installation (A5)

**Distribution (A16):** the factory location is unknown (A03), so the PCR default for international transport is used (S08, §2.5.3, p. 22): 19,000 km by container ship plus 1,000 km by lorry, for the product with its packaging. This is the most conservative of the three PCR defaults.

**Installation (A16):** PSR-0018 counts installation processes and the treatment of packaging waste (S01, §3.2.3). Installation processes are taken as negligible, as in the published PEPs. Packaging waste follows the PSR default (S01, p. 18): for cardboard, wood, and paper, 89% recycled, 8% energy recovery, and 3% split equally between incineration and landfill, plus 1,000 km by lorry.

In [38]:
SHIP_KM, LORRY_KM = 19_000, 1_000            # S08 p. 22, international default (A16)
SHIP_KGCO2E_PER_TKM = 0.01612                 # S19: cargo ship, container ship, average
# HGV_KGCO2E_PER_TKM is defined in Section 4 (S19)

shipped_t = bom["mass_kg"].sum() / 1000       # product + packaging
DISTRIBUTION_KGCO2E = shipped_t * (SHIP_KM * SHIP_KGCO2E_PER_TKM + LORRY_KM * HGV_KGCO2E_PER_TKM)

# Packaging end of life at installation (S01 p. 18) with S19 waste factors (kg CO2e per tonne)
PKG_ROUTES = {"recycling": 0.89, "energy_recovery": 0.08, "incineration": 0.015, "landfill": 0.015}
WASTE_FACTORS_PER_T = {
    "Cardboard": {"recycling": 4.65358, "energy_recovery": 4.65358, "incineration": 4.65358, "landfill": 1164.51317},
    "Paper":     {"recycling": 4.65358, "energy_recovery": 4.65358, "incineration": 4.65358, "landfill": 1164.51317},
    "Wood":      {"recycling": 4.65358, "energy_recovery": 4.65358, "incineration": 4.65358, "landfill": 925.36725},
}
packaging = bom[bom["component"] == "Packaging"].set_index("material")["mass_kg"]
pkg_treatment = sum(packaging[m] / 1000 * share * WASTE_FACTORS_PER_T[m][route]
                    for m in packaging.index for route, share in PKG_ROUTES.items())
pkg_transport = packaging.sum() / 1000 * LORRY_KM * HGV_KGCO2E_PER_TKM
INSTALLATION_KGCO2E = pkg_treatment + pkg_transport

print(f"Distribution: {shipped_t * 1000:.2f} kg shipped -> {DISTRIBUTION_KGCO2E:.2f} kg CO2e")
print(f"Installation: packaging treatment {pkg_treatment:.3f} + transport {pkg_transport:.3f} = {INSTALLATION_KGCO2E:.2f} kg CO2e")

Distribution: 9.18 kg shipped -> 3.76 kg CO2e
Installation: packaging treatment 0.044 + transport 0.212 = 0.26 kg CO2e


## 6. Use Stage (B1–B7)

PSR-0018 (S01, §3.2.4) counts only the charger's own consumption: losses while charging plus its intrinsic consumption. Electricity delivered to the vehicle is excluded (it belongs to the vehicle's footprint).

**Energy (A17):** instead of the PSR formula with European default values, the charger's consumption uses ENERGY STAR measured data for U.S. Level 2 chargers (S21, median of 386 single-output models). Each hour of the 10-year life is assigned to one mode, following the PSR scenario (S01, p. 20):

| Mode | Hours over 10 years | Power (median) |
|---|---|---|
| Charging | 1,040 charges × (27.09 kWh ÷ 11.52 kW) | Full-current total loss |
| Plugged in, not charging | 1,040 × (12 h − charging time) | Idle mode |
| No vehicle | All remaining hours | No-vehicle mode |

**Grid factors (A18):** eGRID2023 state CO₂e output rates (S20), divided by (1 − grid gross loss) so they represent electricity used at the outlet.

In [39]:
LB_TO_KG = 0.45359237
HOURS_10Y = SERVICE_LIFE_YEARS * 8760
CHARGES = 1040                      # S01 p. 20
KWH_PER_CHARGE = 27.09               # S01 p. 20 (1,040 x 27.09 = 28,174 kWh, 0.3% below the 28,251 kWh total; PSR rounding)
PLUGGED_H_PER_CHARGE = 12           # S01 p. 20

# ENERGY STAR measured data (S21): single-output Level 2 chargers
es = pd.read_csv("../data/raw/energystar_evse_ac_2026-10-22.csv")
es = es[(es["Product Type"] == "Level 2") & (es["Number of Outputs"].fillna(1) <= 1)]
col = {"power_kw": "Maximum Output Power (kW)", "loss_w": "Full Current Operation Mode Test: Total Loss (watts)",
       "idle_w": "Idle Mode Input Power (watts)", "novehicle_w": "No Vehicle Mode Input Power (watts)"}
es_med = es[list(col.values())].apply(pd.to_numeric, errors="coerce").median()
es_med.index = list(col.keys())
print(f"ENERGY STAR models used: {len(es)}")
print(es_med.round(2).to_string())

charging_h = CHARGES * KWH_PER_CHARGE / es_med["power_kw"]
idle_h = CHARGES * PLUGGED_H_PER_CHARGE - charging_h
novehicle_h = HOURS_10Y - CHARGES * PLUGGED_H_PER_CHARGE
use_kwh_by_mode = pd.Series({
    "Charging losses": charging_h * es_med["loss_w"] / 1000,
    "Idle (plugged in)": idle_h * es_med["idle_w"] / 1000,
    "No vehicle": novehicle_h * es_med["novehicle_w"] / 1000,
})
USE_KWH = use_kwh_by_mode.sum()
print(use_kwh_by_mode.round(1).to_string())
print(f"Use-stage electricity: {USE_KWH:.1f} kWh over {SERVICE_LIFE_YEARS} years")

# Grid factors (S20): state CO2e output rate (lb/MWh) and grid gross loss
EGRID_LB_PER_MWH = {"California": 394.769, "Texas": 771.179}   # ST23, STC2ERTA
GRID_GROSS_LOSS = {"California": 0.041, "Texas": 0.042}        # GGL23: Western, ERCOT
GRID_EF_KGCO2E_PER_KWH = {s: EGRID_LB_PER_MWH[s] * LB_TO_KG / 1000 / (1 - GRID_GROSS_LOSS[s]) for s in EGRID_LB_PER_MWH}

use_stage = {s: USE_KWH * ef for s, ef in GRID_EF_KGCO2E_PER_KWH.items()}
for s in use_stage:
    print(f"{s}: {GRID_EF_KGCO2E_PER_KWH[s]:.4f} kg CO2e/kWh -> {use_stage[s]:.1f} kg CO2e")

ENERGY STAR models used: 386
power_kw        11.52
loss_w         104.35
idle_w           6.06
novehicle_w      3.52
Charging losses      255.2
Idle (plugged in)     60.8
No vehicle           264.8
Use-stage electricity: 580.8 kWh over 10 years
California: 0.1867 kg CO2e/kWh -> 108.4 kg CO2e
Texas: 0.3651 kg CO2e/kWh -> 212.1 kg CO2e


## 7. End of Life (C1–C4)

PSR-0018 applies the PCR rules (S01, §3.2.5). When the treatment is unknown, the PCR default is transport of 1,000 km by lorry followed by landfill by material family, with PCBs counted as plastics (S08, pp. 24–25). The treatment of this product in the U.S. is unknown, so the default is used (A19). Packaging is excluded here because it is treated at installation (Section 5).

Recycling benefits are reported separately as Module D and are not subtracted from the life cycle total (S08).

In [ ]:
EOL_LORRY_KM = 1000                     # S08 p. 25 default (A19)
# LANDFILL_KGCO2E_PER_T (9.00687, S19) and HGV_KGCO2E_PER_TKM (0.10356, S19) are defined in Section 4

product_t = bom.loc[bom["component"] != "Packaging", "mass_kg"].sum() / 1000
eol_transport = product_t * EOL_LORRY_KM * HGV_KGCO2E_PER_TKM
eol_landfill = product_t * LANDFILL_KGCO2E_PER_T
END_OF_LIFE_KGCO2E = eol_transport + eol_landfill
print(f"End of life: {product_t * 1000:.2f} kg -> transport {eol_transport:.3f} + landfill {eol_landfill:.3f} = {END_OF_LIFE_KGCO2E:.2f} kg CO2e")

## 8. Results

- **Per wallbox (declared unit):** sizes the CDR plan in notebook 04
- **Per kWh (functional unit):** benchmarks against published PEPs in notebook 03
- **Share by stage:** identifies the hotspots

In [41]:
stage_values = {
    "Manufacturing (A1-A3)": manufacturing["kgco2e"].sum() if "kgco2e" in manufacturing else None,
    "Distribution (A4)": DISTRIBUTION_KGCO2E,
    "Installation (A5)": INSTALLATION_KGCO2E,
    "End of life (C1-C4)": END_OF_LIFE_KGCO2E,
}

results = {}
if None not in stage_values.values():
    for state, use_kgco2e in use_stage.items():
        stages = {**stage_values, "Use (B1-B7)": use_kgco2e}
        total = sum(stages.values())
        results[state] = pd.DataFrame(
            {
                "kgco2e_per_wallbox": stages,
                "kgco2e_per_kwh": {k: v * REFERENCE_FLOW for k, v in stages.items()},
                "share": {k: v / total for k, v in stages.items()},
            }
        )
        print(f"{state}: {total:.1f} kg CO2e per wallbox | {total * REFERENCE_FLOW:.2e} kg CO2e per kWh")
results

{}

## 9. Cut-off Compliance

Notebook 01, Section 4: excluded flows must stay at or below 5% of mass, primary energy, and environmental impact in aggregate (S08, §2.3). PCR ed4 Table 2 items cannot be excluded.

- **Mass:** no material is excluded; the BOM mass balance closes (Section 2f).
- **Primary energy:** this study models climate change only, so the energy criterion is checked through its climate impact (a limitation).
- **Climate impact:** three processes are excluded. Each gets a deliberately high upper bound (A20). Assembly energy has no public data, so instead of inventing a value, the notebook reports the assembly energy that would be needed to reach the 5% threshold (break-even). The threshold uses the smaller total (California), and the break-even uses the higher grid factor (Texas), so both are conservative.

In [ ]:
DRAWING_KGCO2E_PER_KG = 3.622 * (1 - 0.93)   # A20: S09 p. 19, semi-fabrication share of wire rod
cable_copper = (bom["component"] == "Charging cable") & (bom["material"] == "Copper")
drawn_copper_kg = bom.loc[cable_copper | (bom["material"] == "Internal wiring"), "mass_kg"].sum()
pa_kg = bom.loc[bom["material"] == "PA", "mass_kg"].sum()

exclusions = pd.DataFrame([
    {"item": "Wire drawing of cable copper and internal wiring", "stage": "Manufacturing", "mass_kg": 0.0,
     "kgco2e": drawn_copper_kg * DRAWING_KGCO2E_PER_KG, "reason": "S09 ends at wire rod (A13); upper bound A20", "source_id": "S09; A13; A20"},
    {"item": "Moulding of PA parts", "stage": "Manufacturing", "mass_kg": 0.0,
     "kgco2e": pa_kg * 4.52, "reason": "S18 ends at pellets; upper bound = PA6 production impact (A20)", "source_id": "S18; A20"},
])

threshold = CUTOFF_THRESHOLD * results["California"]["kgco2e_per_wallbox"].sum()   # smaller total -> stricter
headroom = threshold - exclusions["kgco2e"].sum()
breakeven_kwh = headroom / max(GRID_EF_KGCO2E_PER_KWH.values())                  # higher grid factor -> stricter

print(f"Excluded mass share: 0.0% (threshold {CUTOFF_THRESHOLD:.0%})")
print(f"Excluded impact (bounded items): {exclusions['kgco2e'].sum():.2f} kg CO2e = "
      f"{exclusions['kgco2e'].sum() / results['California']['kgco2e_per_wallbox'].sum():.1%} of the California total")
print(f"5% threshold: {threshold:.2f} kg CO2e -> assembly energy would need to exceed {breakeven_kwh:.0f} kWh per wallbox "
      f"to breach it (A15; S14 reports no assembly energy)")
exclusions.round(3)

## 10. Consistency Check (before moving to notebook 03)

Review this notebook as if you were the assurance reviewer.

- [x] The parameters in Section 0 match notebook 01.
- [x] Every BOM row has a source ID and a data tier, and the mass balance gap is explained.
- [x] All PCR ed4 Table 2 items in the product are included.
- [x] Every emission factor has a source ID and a DQR score.
- [x] Electricity delivered to the vehicle is excluded from the use stage.
- [x] Excluded items stay within the 5% cut-off thresholds (assembly energy below the break-even shown in Section 9).
- [x] Every number is either sourced in `data/raw/SOURCES.md` or logged in `docs/assumptions_log.md`.
- [x] The README status has been updated.

## References

Full details (publisher, URL, version, access date) are in [`data/raw/SOURCES.md`](../data/raw/SOURCES.md). Assumptions A01–A20 are in [`docs/assumptions_log.md`](../docs/assumptions_log.md).

| ID | Source | Used for |
|---|---|---|
| S01 | P.E.P. Association, PSR-0018 ed1.3 (2025-04-08) | Use scenario, use-stage scope, manufacturing waste and packaging defaults, end-of-life rule |
| S04 | ChargePoint Home Flex (CPH50) datasheet | Product mass cross-check (A04) |
| S05 | Tesla Universal Wall Connector datasheet | Product mass cross-check (A04) |
| S06 | Wallbox Pulsar Plus North America datasheet | Body mass without cable (A04) |
| S07 | ENERGY STAR listing, Grizzl-E Classic | Cable gauge (A05) |
| S08 | P.E.P. Association, PEP-PCR ed4 (2021-09-06) | Cut-off, DQR method (Annex E), transport and end-of-life defaults |
| S09 | International Copper Association, semi-fabricated products LCA | Copper factor; wire drawing bound |
| S10–S14 | PEPs of five attached-cable wallboxes (Legrand ×2, Ensto, Hager, Qovoltis) | Body and packaging composition; manufacturing and per-kWh benchmarks |
| S13 | Hager witty One PEP | Cable jacket mass per metre (A05) |
| S15 | PEP ecopassport registry | Screening of comparable wallboxes |
| S16 | Delta AC MAX NA-Smart manual | Product mass reference point only (A04) |
| S17 | Phoenix Contact J1772 connector datasheet | Connector mass (A06) |
| S18 | PlasticsEurope PA6 eco-profile | PA factor |
| S19 | UK DESNZ GHG conversion factors 2026 | Material, transport, and waste factors |
| S20 | U.S. EPA eGRID2023 | Grid emission rates and grid loss, California and Texas |
| S21 | ENERGY STAR certified AC EVSE dataset | Use-stage losses and standby power |

Secondary data notes: S02 (PSR annex calculator) and S03 (Hager socket PEP) are not used in this notebook.